# 🏆 Benchmark Challenge: Agentic Financial RAG & Deterministic Tool-Calling (UTN FRSR)
## Plantilla Inicial para Google Colab (Microservicio Caja Negra + Cloudflare Tunnel)

Este notebook prepara tu entorno en Google Colab, levanta el servidor FastAPI (`src/server.py`) en un hilo dentro del mismo kernel (para compartir la GPU T4 si montás un modelo local) y expone la URL pública mediante **Cloudflare Tunnel (`cloudflared`)**.

In [ ]:
#@title 1. Clonar Starter Kit, instalar dependencias y descargar cloudflared
import os
from pathlib import Path

if not Path("src/schemas.py").exists():
    if not Path("/content/aieng-utn-rag-challenge").exists():
        !git clone -q https://github.com/chebaperez/aieng-utn-rag-challenge.git /content/aieng-utn-rag-challenge
    %cd /content/aieng-utn-rag-challenge

!pip install -q -r requirements.txt

if not os.path.exists("/usr/local/bin/cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

print("✅ Entorno listo.")

In [ ]:
#@title 2. Levantar servidor FastAPI (en hilo), probar evaluación local y abrir Cloudflare Tunnel
import re, subprocess, threading, time
import httpx, uvicorn
from src.server import app

config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning")
uvicorn_server = uvicorn.Server(config)
threading.Thread(target=uvicorn_server.run, daemon=True).start()
time.sleep(2.0)

print("Healthcheck local:", httpx.get("http://127.0.0.1:8000/health", timeout=5.0).json())

# Prueba rápida contra 5 casos de data/dev_set.json
!python scripts/run_local_eval.py --limit 5

# Abrir túnel público para el Evaluador Remoto
tunnel_proc = subprocess.Popen(
    ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
for line in tunnel_proc.stdout:
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if m:
        print("\n======================================================")
        print("🌐 URL PÚBLICA PARA EL EVALUADOR REMOTO:")
        print(m.group(0))
        print("======================================================\n")
        break